## **Step 1: Initialize a Database**

In [1]:
import sqlite3

DB_PATH = "./db/sol-2/invoices.db"

# Initialize DB connection
def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS invoices (
            file TEXT PRIMARY KEY UNIQUE,
            client TEXT,
            amount REAL,
            product TEXT
        )
    """)
    conn.commit()
    return conn

## **Step 2: Initialize Tools**

- read_invoice_file(file_path)
- save_invoice_to_db(file, client, amount, product)

Idea is to load tool from an MCP server:
- https://developer.box.com/guides/box-mcp/tools

In [2]:
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader

@tool
def read_invoice_file(file_path: str) -> str:
    """Reads the text content of a PDF invoice file."""
    # Note: Use PyPDFLoader here as shown in the previous step
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    return "\n".join([d.page_content for d in docs])

In [3]:
import sqlite3

@tool
def save_invoice_to_db(file: str, client: str, amount: float, product: str) -> str:
    """Saves extracted invoice data to the SQL database."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO invoices (file, client, amount, product)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(file) DO UPDATE SET
            client=excluded.client,
            amount=excluded.amount,
            product=excluded.product
    """, (file, client, amount, product))
    conn.commit()
    conn.close()
    return "Database updated successfully."

## **Step 3: Configure the Agent**

- Agent 1: This agent will list the IDs of all files inside a given folder.
- Agent 2: This agent will extract invoice data from a file in Box. You'll instruct the agent to extract the client name, invoice amount, and product name, and return everything as a JSON object in a specific format.
- Agent 3: This agent will coordinate between the other agents to process every invoice in Box and extract their data.

In [4]:
from pydantic import BaseModel, Field
from typing import Optional

class InvoiceData(BaseModel):
    client_name: Optional[str] = Field(description="Client name")
    invoice_amount: Optional[float] = Field(description="Invoice amount")
    product_name: Optional[str] = Field(description="Product name")

In [5]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0.0
)

llm.invoke("hi").content

'Hello! How can I assist you today?'

In [6]:
from langchain.agents import create_agent

tools = [read_invoice_file, save_invoice_to_db]

agent_executor = create_agent(
    model=llm, 
    tools=tools,
    response_format=InvoiceData,
    system_prompt="You are an invoice processing expert. Read the file, extract client, amount, and product, then save to the database.",
)

## **Step 4: Run the Processing Loop**

Iterate through your local folder and invoke the agent for each file

In [7]:
import os

init_db()
INVOICE_DIR = "./invoices"

for filename in os.listdir(INVOICE_DIR):
    if filename.endswith(".pdf"):
        file_path = os.path.join(INVOICE_DIR, filename)
        
        # Invoke agent
        query = f"Process the file at {file_path} and save the data to the database."
        response = agent_executor.invoke({"messages": [("user", query)]})
        print(f"Finished processing: {filename}")

Ignoring wrong pointing object 2 65536 (offset 0)


Finished processing: demo-invoice-no-tax-6.pdf


Ignoring wrong pointing object 2 65536 (offset 0)


Finished processing: demo-invoice-20tax-2.pdf


Ignoring wrong pointing object 2 65536 (offset 0)
Ignoring wrong pointing object 22 65536 (offset 0)


Finished processing: demo-invoice-20tax-9.pdf


Ignoring wrong pointing object 2 65536 (offset 0)


Finished processing: demo-invoice-no-tax-8.pdf


Ignoring wrong pointing object 2 65536 (offset 0)
Ignoring wrong pointing object 22 65536 (offset 0)


Finished processing: demo-invoice-no-tax-9.pdf


## **Step 5: Generate the Final Report**

In [9]:
# Generating Final Report
print("\nInvoice Report")

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*), SUM(amount) FROM invoices")
total_invoices, total_amount = cursor.fetchone()

print(f"* Total invoices: {total_invoices}")
print(f"* Total amount: {total_amount}")

print("\nBreakdown by client:")
cursor.execute("SELECT client, COUNT(*), SUM(amount) FROM invoices GROUP BY client")
for row in cursor.fetchall():
    client, count, amount = row
    print(f"* {client}: {count} invoices (${amount})")

conn.close()


Invoice Report
* Total invoices: 5
* Total amount: 584757.99

Breakdown by client:
* ACME Inc: 5 invoices ($584757.99)
